In [1]:
import json
import numpy as np
import pyvisa
import time
import hid # pip install hidapi
import skrf as rf
import matplotlib.pyplot as plt
import serial.tools.list_ports
from windfreak import SynthHD
from RsInstrument import RsInstrument, BinFloatFormat

rm = pyvisa.ResourceManager()
instruments = rm.list_resources()

In [2]:
spa = None
for instrument in instruments:
    if '7405' in instrument:
        spa = rm.open_resource(instrument) 
        break

In [11]:
print(spa.query('*IDN?'))

Hewlett-Packard, E7405A, SG45102671, A.14.04



In [12]:
spa_json = {}
spa_json['idn'] = spa.query('*IDN?').strip()
spa_json['fstart']    = float(spa.query(':SENS1:FREQ:START?'))
spa_json['fstop']     = float(spa.query(':SENS1:FREQ:STOP?'))
spa_json['numpoints'] = int(spa.query(':SENS1:SWEep:POINts?'))
spa_json['rbw']      = float(spa.query(':SENS:BAND?'))
spa_json['vbw'] = float(spa.query(':SENS:BAND:VID?'))
spa_json['unit'] = spa.query(":UNIT:POW?").strip()
spa.write(":SYST:LOCAL")
spa_json

{'idn': 'Hewlett-Packard, E7405A, SG45102671, A.14.04',
 'fstart': 6000000000.0,
 'fstop': 6250000000.0,
 'numpoints': 401,
 'rbw': 1000000.0,
 'vbw': 3000.0,
 'unit': 'W'}

In [13]:
frequencies = np.linspace(spa_json['fstart'],spa_json['fstop'],spa_json['numpoints'])
fghz = frequencies/1e9

In [6]:
spa.query(":UNIT:POW?")


'W\n'

In [7]:
spa.write(":UNIT:POW W")
spa.write("FORM REAL,32")
spa.write("INIT:CONT OFF")
spa.timeout = 30000
spa.write("INIT:IMM")
spa.query("*OPC?")
spa.timeout = 5000
trace_data = spa.query_binary_values("TRAC? TRACE1", datatype="f", is_big_endian=True)
spa.write("INIT:CONT ON")




[3.307501147964409e-11,
 3.360469888469275e-11,
 3.21218121546174e-11,
 3.5285802058027116e-11,
 3.351197444545484e-11,
 3.0436868303507225e-11,
 3.250124475107086e-11,
 3.345029808698996e-11,
 3.452232250067411e-11,
 3.498646164445951e-11,
 3.250872834814622e-11,
 3.518033780958163e-11,
 3.4641765150889015e-11,
 3.199631878891829e-11,
 3.290031094782542e-11,
 3.3674410482298356e-11,
 3.1644627890292654e-11,
 3.2621204348881605e-11,
 3.57108335957701e-11,
 3.518033780958163e-11,
 3.520464475492702e-11,
 3.575197082827941e-11,
 3.051406002874124e-11,
 3.16154949442371e-11,
 3.319709090954248e-11,
 3.608278259514819e-11,
 3.248628102636708e-11,
 3.157184583213457e-11,
 3.179069507641685e-11,
 3.354285252332723e-11,
 3.326595596209181e-11,
 3.131843395731693e-11,
 3.228494208085131e-11,
 3.422945607511885e-11,
 3.255367156396183e-11,
 3.195214232087906e-11,
 3.113866456350145e-11,
 3.4158596090572146e-11,
 3.2218106654768874e-11,
 3.3227677553870905e-11,
 3.1311220977103815e-11,
 3.259867

In [8]:
type(trace_data)

list

In [10]:
spa

<'USBInstrument'('USB0::0x03EB::0x2065::Hewlett-Packard__E7405A__SG45102671__A.14.04::0::INSTR')>